In [28]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import wandb
import random

from datasets import load_dataset
import evaluate
from dataclasses import dataclass, asdict
from transformers import (
    AutoTokenizer,
    AutoModel,
    BartTokenizer,
    BartForConditionalGeneration,
)
from transformers.modeling_outputs import BaseModelOutput

# =====================
# Config
# =====================
@dataclass
class TrainingConfig:
    batch_size: int = 8   # updated
    lr: float = 2e-5
    num_epochs: int = 10  # updated
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    log_interval: int = 10
    max_len: int = 512

config = TrainingConfig()

# =====================
# Init WandB
# =====================
wandb.init(
    project="Prot",
    config=asdict(config),
    name="bart_multimodal_run"
)

# =====================
# Dataset
# =====================
dataset = load_dataset("vladak/drug_protein_mechanism")

# =====================
# Tokenizers
# =====================
chem_tokenizer = AutoTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
prot_tokenizer = AutoTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

# =====================
# Models
# =====================
chem_model = AutoModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1").to(config.device)
prot_model = AutoModel.from_pretrained("Rostlab/prot_bert").to(config.device)
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to(config.device)

fusion_dim = bart_model.config.d_model
fusion_layer = nn.Linear(
    chem_model.config.hidden_size + prot_model.config.hidden_size,
    fusion_dim
).to(config.device)

# =====================
# Dataset processing
# =====================
def collate_fn(batch):
    smiles = [x["drug_smiles"] for x in batch]
    prots = [x["target_sequence"] for x in batch]
    texts = [x["mechanistic_explanation"] for x in batch]

    chem_enc = chem_tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)
    prot_enc = prot_tokenizer(prots, return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)
    text_enc = bart_tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)

    return chem_enc, prot_enc, text_enc, texts

train_loader = DataLoader(dataset["train"], batch_size=config.batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(dataset["val"], batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(dataset["test"], batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn)

# =====================
# Optimizer
# =====================
optimizer = torch.optim.AdamW(
    list(fusion_layer.parameters()) + list(bart_model.parameters()), 
    lr=config.lr
)

# =====================
# Metrics
# =====================
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def compute_text_metrics(preds, refs):
    results = {}
    bleu = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    rouge = rouge_metric.compute(predictions=preds, references=refs)
    bert = bertscore_metric.compute(predictions=preds, references=refs, lang="en")

    results["bleu"] = bleu["score"]
    results["rougeL"] = rouge["rougeL"]
    results["bertscore"] = sum(bert["f1"]) / len(bert["f1"])
    return results


# =====================
# Pick 4 fixed validation sample indices
# =====================
num_val_samples = len(dataset["val"])
fixed_val_indices = random.sample(range(num_val_samples), 4)

# =====================
# Training Loop
# =====================
for epoch in range(config.num_epochs):
    bart_model.train()
    total_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}")

    for step, (chem_enc, prot_enc, text_enc, texts) in enumerate(progress):
        optimizer.zero_grad()

        # Encode SMILES
        chem_out = chem_model(**{k: v.to(config.device) for k, v in chem_enc.items()})
        chem_emb = chem_out.last_hidden_state.mean(dim=1)

        # Encode Protein
        prot_out = prot_model(**{k: v.to(config.device) for k, v in prot_enc.items()})
        prot_emb = prot_out.last_hidden_state.mean(dim=1)

        # Fuse
        fused = fusion_layer(torch.cat([chem_emb, prot_emb], dim=1))

        # Wrap in BaseModelOutput for BART
        encoder_hidden_states = fused.unsqueeze(1).expand(fused.size(0), 10, -1)
        encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)

        labels = text_enc["input_ids"].to(config.device)

        outputs = bart_model(
            encoder_outputs=encoder_outputs,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if step % config.log_interval == 0:
            avg_loss = total_loss / (step + 1)
            progress.set_postfix({"loss": avg_loss})
            wandb.log({"train_loss": avg_loss, "epoch": epoch+1, "step": step})

    # =====================
    # Validation
    # =====================
    bart_model.eval()
    preds, refs = [], []

    with torch.no_grad():
        all_val_samples = []
        for chem_enc, prot_enc, text_enc, texts in tqdm(val_loader, desc="Validation"):
            chem_out = chem_model(**{k: v.to(config.device) for k, v in chem_enc.items()})
            chem_emb = chem_out.last_hidden_state.mean(dim=1)

            prot_out = prot_model(**{k: v.to(config.device) for k, v in prot_enc.items()})
            prot_emb = prot_out.last_hidden_state.mean(dim=1)

            fused = fusion_layer(torch.cat([chem_emb, prot_emb], dim=1))
            encoder_hidden_states = fused.unsqueeze(1).expand(fused.size(0), 10, -1)
            encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)

            generated = bart_model.generate(
                encoder_outputs=encoder_outputs,
                max_length=config.max_len
            )
            decoded = bart_tokenizer.batch_decode(generated, skip_special_tokens=True)

            preds.extend(decoded)
            refs.extend(texts)
            all_val_samples.extend(list(zip(texts, decoded)))

        # Compute metrics
        metrics = compute_text_metrics(preds, refs)
        print(f"Validation metrics: {metrics}")
        wandb.log({f"val_{k}": v for k, v in metrics.items()})

        # Print fixed 4 validation samples predictions
        print("\nValidation sample predictions:")
        for idx in fixed_val_indices:
            sample = dataset["val"][idx]
            # encode & predict this sample only
            chem_enc = chem_tokenizer([sample["drug_smiles"]], return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)
            prot_enc = prot_tokenizer([sample["target_sequence"]], return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)
            chem_out = chem_model(**{k: v.to(config.device) for k, v in chem_enc.items()})
            chem_emb = chem_out.last_hidden_state.mean(dim=1)
            prot_out = prot_model(**{k: v.to(config.device) for k, v in prot_enc.items()})
            prot_emb = prot_out.last_hidden_state.mean(dim=1)
            fused = fusion_layer(torch.cat([chem_emb, prot_emb], dim=1))
            encoder_hidden_states = fused.unsqueeze(1).expand(fused.size(0), 10, -1)
            encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)

            generated = bart_model.generate(encoder_outputs=encoder_outputs, max_length=config.max_len)
            pred_text = bart_tokenizer.decode(generated[0], skip_special_tokens=True)

            print("REF: ", sample["mechanistic_explanation"])
            print("PRED:", pred_text)
            print("-" * 80)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Validation: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:14<00:00,  1.49s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.po

Validation metrics: {'bleu': 8.575498535929903, 'rougeL': 0.2476389975112635, 'bertscore': 0.8420175698242689}

Validation sample predictions:
REF:  From biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below: 2HG2‐hydroxyglutarateABCATP‐binding cassette subfamilyACAAacetyl‐CoA acyltransferasesACOaconitasesACTactin superfamilyALDHaldehyde dehydrogenase family,ANXAannexinsAPOapolipoproteinARHGAPrho GTPase‐activating proteins.
PRED: The drug DRUG interacts with its target TARGET (TARGET) in the form of TARGET2. The drug interacts with TARGET3.
--------------------------------------------------------------------------------
REF:  DRUG is a bioactive amine involved in various physiological processes, including immune responses, inflammation, and neurotransmission. It is synthesized from histidine via histidine decarboxylase (HDC) and exerts

Validation: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:12<00:00,  1.28s/it]


Validation metrics: {'bleu': 55.38856723890833, 'rougeL': 0.6601181858513738, 'bertscore': 0.933158004754468}

Validation sample predictions:
REF:  From biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below: 2HG2‐hydroxyglutarateABCATP‐binding cassette subfamilyACAAacetyl‐CoA acyltransferasesACOaconitasesACTactin superfamilyALDHaldehyde dehydrogenase family,ANXAannexinsAPOapolipoproteinARHGAPrho GTPase‐activating proteins.
PRED: The drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below: “TARGET 2”. TARGET 2 is a protein-coupled gene-transfer network.
--------------------------------------------------------------------------------
REF:  DRUG is a bioactive amine involved in various physiological processes, including immune responses, inflammation, and neurotransmission. It is synthesized

Validation: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:14<00:00,  1.44s/it]


...

Validation metrics: {'bleu': 80.62380662693714, 'rougeL': 0.8339555826207611, 'bertscore': 0.9658250981255582}

Validation sample predictions:
REF:  From biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below: 2HG2‐hydroxyglutarateABCATP‐binding cassette subfamilyACAAacetyl‐CoA acyltransferasesACOaconitasesACTactin superfamilyALDHaldehyde dehydrogenase family,ANXAannexinsAPOapolipoproteinARHGAPrho GTPase‐activating proteins.
PRED: From biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below: 2HG2‐hydroxyglutarateABCATP‐binding cassette subfamilyACAAacetyl‐CoA acyltransferasesACOaconitasesACTactin superfamilyALDHaldehyde dehydrogenase family,ANXAannexinsAPOapolipoproteinARHGAPrho GTPase‐ac

Validation: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:15<00:00,  1.56s/it]


Validation metrics: {'bleu': 78.08527516232724, 'rougeL': 0.8267813033853526, 'bertscore': 0.9644715425215269}

Validation sample predictions:
REF:  From biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below: 2HG2‐hydroxyglutarateABCATP‐binding cassette subfamilyACAAacetyl‐CoA acyltransferasesACOaconitasesACTactin superfamilyALDHaldehyde dehydrogenase family,ANXAannexinsAPOapolipoproteinARHGAPrho GTPase‐activating proteins.
PRED: From biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below: 2HG2‐hydroxyglutarateABCATP‐binding cassette subfamilyACAAacetyl‐CoA acyltransferasesACOaconitasesACTactin superfamilyALDHaldehyde dehydrogenase family,ANXAannexinsAPOapolipoproteinARHGAPrho GTPase‐ac

Validation: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:15<00:00,  1.55s/it]


Validation metrics: {'bleu': 77.44067215201318, 'rougeL': 0.8151881627343034, 'bertscore': 0.9625606599606966}

Validation sample predictions:
REF:  From biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below: 2HG2‐hydroxyglutarateABCATP‐binding cassette subfamilyACAAacetyl‐CoA acyltransferasesACOaconitasesACTactin superfamilyALDHaldehyde dehydrogenase family,ANXAannexinsAPOapolipoproteinARHGAPrho GTPase‐activating proteins.
PRED: From biomedical text below, extract, summarize and consolidate a concise mechanistic explanation of how the drug DRUG interacts with its target TARGET (TARGET) Keep only facts and information from the text below: 2HG2‐hydroxyglutarateABCATP‐binding cassette subfamilyACAAacetyl‐CoA acyltransferasesACOaconitasesACTactin superfamilyALDHaldehyde dehydrogenase family,ANXAannexinsAPOapolipoproteinARHGAPrho GTPase‐ac

In [ ]:
# import torch
# import torch.nn as nn
# from torch.utils.data import DataLoader
# from tqdm import tqdm
# import wandb

# from datasets import load_dataset
# import evaluate
# from dataclasses import dataclass, asdict
# from transformers import (
#     AutoTokenizer,
#     AutoModel,
#     BartTokenizer,
#     BartForConditionalGeneration,
# )
# from transformers.modeling_outputs import BaseModelOutput

# # =====================
# # Config
# # =====================
# @dataclass
# class TrainingConfig:
#     batch_size: int = 4
#     lr: float = 2e-5
#     num_epochs: int = 3
#     device: str = "cuda" if torch.cuda.is_available() else "cpu"
#     log_interval: int = 10
#     max_len: int = 512

# config = TrainingConfig()

# # =====================
# # Init WandB
# # =====================
# wandb.init(
#     project="Prot",
#     config=asdict(config),
#     name="bart_multimodal_run"
# )

# # =====================
# # Dataset
# # =====================
# dataset = load_dataset("vladak/drug_protein_mechanism")

# # =====================
# # Tokenizers
# # =====================
# chem_tokenizer = AutoTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
# prot_tokenizer = AutoTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
# bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

# # =====================
# # Models
# # =====================
# chem_model = AutoModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1").to(config.device)
# prot_model = AutoModel.from_pretrained("Rostlab/prot_bert").to(config.device)
# bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to(config.device)

# fusion_dim = bart_model.config.d_model
# fusion_layer = nn.Linear(
#     chem_model.config.hidden_size + prot_model.config.hidden_size,
#     fusion_dim
# ).to(config.device)

# # =====================
# # Dataset processing
# # =====================
# def collate_fn(batch):
#     smiles = [x["drug_smiles"] for x in batch]
#     prots = [x["target_sequence"] for x in batch]
#     texts = [x["mechanistic_explanation"] for x in batch]

#     chem_enc = chem_tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)
#     prot_enc = prot_tokenizer(prots, return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)
#     text_enc = bart_tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=config.max_len)

#     return chem_enc, prot_enc, text_enc, texts

# train_loader = DataLoader(dataset["train"], batch_size=config.batch_size, shuffle=True, collate_fn=collate_fn)
# val_loader = DataLoader(dataset["val"], batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn)
# test_loader = DataLoader(dataset["test"], batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn)

# # =====================
# # Optimizer
# # =====================
# optimizer = torch.optim.AdamW(
#     list(fusion_layer.parameters()) + list(bart_model.parameters()), 
#     lr=config.lr
# )

# # =====================
# # Metrics
# # =====================
# bleu_metric = evaluate.load("sacrebleu")
# rouge_metric = evaluate.load("rouge")
# bertscore_metric = evaluate.load("bertscore")

# def compute_text_metrics(preds, refs):
#     results = {}
#     bleu = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
#     rouge = rouge_metric.compute(predictions=preds, references=refs)
#     bert = bertscore_metric.compute(predictions=preds, references=refs, lang="en")

#     results["bleu"] = bleu["score"]
#     results["rougeL"] = rouge["rougeL"]
#     results["bertscore"] = sum(bert["f1"]) / len(bert["f1"])
#     return results

# # =====================
# # Training Loop
# # =====================
# for epoch in range(config.num_epochs):
#     bart_model.train()
#     total_loss = 0.0
#     progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}")

#     for step, (chem_enc, prot_enc, text_enc, texts) in enumerate(progress):
#         optimizer.zero_grad()

#         # Encode SMILES
#         chem_out = chem_model(**{k: v.to(config.device) for k, v in chem_enc.items()})
#         chem_emb = chem_out.last_hidden_state.mean(dim=1)

#         # Encode Protein
#         prot_out = prot_model(**{k: v.to(config.device) for k, v in prot_enc.items()})
#         prot_emb = prot_out.last_hidden_state.mean(dim=1)

#         # Fuse
#         fused = fusion_layer(torch.cat([chem_emb, prot_emb], dim=1))

#         # Wrap in BaseModelOutput for BART
#         encoder_hidden_states = fused.unsqueeze(1).expand(fused.size(0), 10, -1)
#         encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)

#         labels = text_enc["input_ids"].to(config.device)

#         outputs = bart_model(
#             encoder_outputs=encoder_outputs,
#             labels=labels
#         )

#         loss = outputs.loss
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()
#         if step % config.log_interval == 0:
#             avg_loss = total_loss / (step + 1)
#             progress.set_postfix({"loss": avg_loss})
#             wandb.log({"train_loss": avg_loss, "epoch": epoch+1, "step": step})

#     # =====================
#     # Validation
#     # =====================
#     bart_model.eval()
#     preds, refs = [], []
#     with torch.no_grad():
#         for chem_enc, prot_enc, text_enc, texts in tqdm(val_loader, desc="Validation"):
#             chem_out = chem_model(**{k: v.to(config.device) for k, v in chem_enc.items()})
#             chem_emb = chem_out.last_hidden_state.mean(dim=1)

#             prot_out = prot_model(**{k: v.to(config.device) for k, v in prot_enc.items()})
#             prot_emb = prot_out.last_hidden_state.mean(dim=1)

#             fused = fusion_layer(torch.cat([chem_emb, prot_emb], dim=1))
#             encoder_hidden_states = fused.unsqueeze(1).expand(fused.size(0), 10, -1)
#             encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)

#             generated = bart_model.generate(
#                 encoder_outputs=encoder_outputs,
#                 max_length=config.max_len
#             )
#             decoded = bart_tokenizer.batch_decode(generated, skip_special_tokens=True)

#             preds.extend(decoded)
#             refs.extend(texts)

#     metrics = compute_text_metrics(preds, refs)
#     print(f"Validation metrics: {metrics}")
#     wandb.log({f"val_{k}": v for k, v in metrics.items()})

# # =====================
# # Test
# # =====================
# bart_model.eval()
# preds, refs = [], []
# with torch.no_grad():
#     for chem_enc, prot_enc, text_enc, texts in tqdm(test_loader, desc="Testing"):
#         chem_out = chem_model(**{k: v.to(config.device) for k, v in chem_enc.items()})
#         chem_emb = chem_out.last_hidden_state.mean(dim=1)

#         prot_out = prot_model(**{k: v.to(config.device) for k, v in prot_enc.items()})
#         prot_emb = prot_out.last_hidden_state.mean(dim=1)

#         fused = fusion_layer(torch.cat([chem_emb, prot_emb], dim=1))
#         encoder_hidden_states = fused.unsqueeze(1).expand(fused.size(0), 10, -1)
#         encoder_outputs = BaseModelOutput(last_hidden_state=encoder_hidden_states)

#         generated = bart_model.generate(
#             encoder_outputs=encoder_outputs,
#             max_length=config.max_len
#         )
#         decoded = bart_tokenizer.batch_decode(generated, skip_special_tokens=True)

#         preds.extend(decoded)
#         refs.extend(texts)

# metrics = compute_text_metrics(preds, refs)
# print(f"Test metrics: {metrics}")
# wandb.log({f"test_{k}": v for k, v in metrics.items()})
